# Stage 5: Model Selection and Development
## CDS 4003 – Hotel Booking Cancellation Prediction

## 5.1 Introduction

**Goal of this stage:** compare several machine learning models for predicting whether a hotel booking will be cancelled (`is_canceled`), explain how each model works, and justify the final choice.

**Business framing.** A cancelled room that is not resold is lost revenue. A model that flags bookings likely to cancel lets a hotel manage overbooking, send targeted reminders and plan staffing. This is a **binary classification** problem (0 = kept, 1 = cancelled).

**Approach**
1. Review the Stage 4 dataset for **data leakage** and duplicate problems before modelling.
2. Build a leakage-safe preprocessing pipeline (fitted on training data only).
3. Split the data once (80/20, stratified) and keep the test set untouched until the end.
4. Compare a **baseline** and four models using **5-fold stratified cross-validation** on the training set.
5. Evaluate all models on the held-out test set, inspect errors and feature importance.
6. Check robustness with a time-based split, then justify the final model.

Hyperparameter tuning is left for Stage 6, so all models here use sensible, near-default settings.

In [1]:
import warnings, time, joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv("hotel_bookings_model_ready.csv")
print("Model-ready dataset from Stage 4:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("\nTarget distribution (%):")
print((df["is_canceled"].value_counts(normalize=True) * 100).round(2))

FileNotFoundError: [Errno 2] No such file or directory: 'hotel_bookings_model_ready.csv'

## 5.2 Pre-modelling Checks: Leakage and Duplicates

Stage 4 already removed `reservation_status` and `reservation_status_date`, which reveal the outcome directly. Before modelling, two more issues need checking because they can make results look better than they really are.

**(a) `assigned_room_type`.** The room is assigned at or near check-in, so it is partly decided *after* the cancellation outcome. A cancelled booking never gets a physical room, which is why some values behave like the target. We check this below.

**(b) Duplicate rows.** Stage 3 kept duplicates because there is no booking ID. For modelling this causes a problem: if identical rows land in both the training and test sets, the model is tested on bookings it has already memorised.

In [ ]:
# (a) Does assigned_room_type behave like the target?
room = df.groupby("assigned_room_type")["is_canceled"].agg(bookings="count", cancel_rate="mean")
room["cancel_rate"] = (room["cancel_rate"] * 100).round(1)
display(room)

# Bookings where the assigned room differs from the reserved room
df["_room_changed"] = (df["assigned_room_type"] != df["reserved_room_type"]).astype(int)
print("\nCancellation rate when room changed vs unchanged (%):")
print((df.groupby("_room_changed")["is_canceled"].mean() * 100).round(1))
df = df.drop(columns="_room_changed")

# (b) Duplicates
n_dup = df.duplicated().sum()
print(f"\nDuplicate rows: {n_dup} ({n_dup/len(df)*100:.1f}% of the data)")

**Findings.** Bookings where the assigned room differs from the reserved room are cancelled only 5.4% of the time, versus 41.6% when the rooms match, and rooms `I` and `K` are almost never cancelled. This is a leakage signature: the hotel records a different assigned room mainly for stays that actually happen (a cancelled guest never receives a room). `assigned_room_type` is therefore **dropped**. `reserved_room_type` (known at booking time) is kept.

Next, we measure how much the duplicates inflate performance, using the same Random Forest on (1) the data with duplicates and (2) the de-duplicated data.

In [ ]:
def prepare_features(data):
    """Deterministic feature preparation (no statistics learned from the data,
    so it is safe to apply before the train/test split)."""
    d = data.copy()
    d = d.drop(columns=["assigned_room_type"])               # leakage (see 5.2)
    d["has_company"] = (d["company"] > 0).astype(int)        # company is an ID, keep only whether one exists
    d = d.drop(columns="company")
    d["agent"] = d["agent"].astype(int).astype(str)          # agent is an ID, so treat it as a category
    d["children"] = d["children"].astype(int)
    d["total_guests"] = d["total_guests"].astype(int)
    return d

CAT_COLS = ["hotel", "arrival_date_month", "meal", "country", "market_segment",
            "distribution_channel", "reserved_room_type", "deposit_type",
            "customer_type", "agent"]

def split_xy(d):
    return d.drop(columns="is_canceled"), d["is_canceled"]

def quick_rf_auc(d):
    X, y = split_xy(d)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=100), CAT_COLS)],
                            remainder="passthrough")
    rf = Pipeline([("pre", pre), ("m", RandomForestClassifier(n_estimators=60, n_jobs=-1, random_state=RANDOM_STATE))])
    rf.fit(Xtr, ytr)
    p = rf.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, p), accuracy_score(yte, rf.predict(Xte)), len(d)

d_all = prepare_features(df)
d_dedup = d_all.drop_duplicates().reset_index(drop=True)

res = []
for name, d in [("With duplicates (Stage 3 decision)", d_all), ("Duplicates removed", d_dedup)]:
    auc, acc, n = quick_rf_auc(d)
    res.append({"Dataset": name, "Rows": n, "Test ROC-AUC": round(auc, 4), "Test Accuracy": round(acc, 4)})
display(pd.DataFrame(res))

**Decision.** Duplicates make the model look better because the same rows appear in both the training and test sets. To get an honest estimate of performance on *new* bookings, we **remove duplicates for modelling only**. This does not contradict Stage 3: identical rows may well be separate reservations, but when they are identical in every feature the model gains nothing from seeing them twice and the test score is inflated. The cost is that the rows removed are genuinely repeated patterns, so the real-world class mix is slightly different. We report this limitation in the final write-up.

In [ ]:
data = d_dedup.copy()
print("Modelling dataset:", data.shape)
print("Cancellation rate: %.2f%%" % (data["is_canceled"].mean() * 100))

X, y = split_xy(data)
NUM_COLS = [c for c in X.columns if c not in CAT_COLS]
print("\nCategorical features (%d): %s" % (len(CAT_COLS), CAT_COLS))
print("Numeric features (%d): %s" % (len(NUM_COLS), NUM_COLS))

## 5.3 Train/Test Split and Validation Strategy

- **80% training / 20% test**, stratified on `is_canceled` so both sets keep the same cancellation rate. The test set is used **once**, at the end.
- **5-fold stratified cross-validation** on the training set is used to compare models. This gives a mean and a spread for each metric, which is more reliable than a single split.
- All preprocessing (scaling, one-hot encoding) sits inside a scikit-learn `Pipeline`, so it is re-fitted on each training fold and **never sees validation or test data**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Cancellation rate - train: %.2f%%, test: %.2f%%" % (y_train.mean()*100, y_test.mean()*100))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 5.4 Preprocessing

| Feature type | Treatment | Reason |
|---|---|---|
| Categorical (10) | One-hot encoding; categories seen fewer than 100 times are grouped into one "infrequent" level | `country` (~170 values) and `agent` (~330 IDs) would otherwise create hundreds of sparse columns |
| Numeric | Standardised for Logistic Regression only | Linear models are sensitive to feature scale; tree-based models are not |

Unseen categories at prediction time are ignored rather than causing an error (`handle_unknown="ignore"`), which matters for the Streamlit app in Stage 7.

In [ ]:
def make_preprocessor(scale_numeric):
    num = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=100, sparse_output=False), CAT_COLS),
        ("num", num, NUM_COLS),
    ])

models = {
    "Baseline (majority class)": Pipeline([("pre", make_preprocessor(False)),
        ("model", DummyClassifier(strategy="most_frequent"))]),
    "Logistic Regression": Pipeline([("pre", make_preprocessor(True)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    "Decision Tree": Pipeline([("pre", make_preprocessor(False)),
        ("model", DecisionTreeClassifier(max_depth=10, min_samples_leaf=20, random_state=RANDOM_STATE))]),
    "Random Forest": Pipeline([("pre", make_preprocessor(False)),
        ("model", RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=RANDOM_STATE))]),
    "Gradient Boosting (HistGB)": Pipeline([("pre", make_preprocessor(False)),
        ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
}
print("Models to compare:", list(models))

## 5.5 The Models: How They Work and Why They Fit

**Baseline: majority class.** Always predicts the most common outcome ("not cancelled"). It is not a real model; it sets the floor that every other model must beat. Its accuracy equals the share of the majority class, so it shows why accuracy alone is a weak metric here.

**Logistic Regression.** Learns one weight per feature and combines them into a probability using the logistic (S-shaped) function. *Strengths:* fast, stable, and each coefficient can be explained to hotel managers ("longer lead time raises cancellation odds"). *Weakness:* assumes a linear effect, so it misses interactions such as "non-refundable *and* long lead time".

**Decision Tree.** Repeatedly splits the data on the feature and threshold that best separates cancelled from kept bookings, producing a readable set of if/then rules. *Strengths:* easy to explain, handles non-linear effects and interactions. *Weakness:* a single tree is unstable and overfits easily, so we limit depth and leaf size.

**Random Forest.** Trains many trees on random samples of rows and features and averages their votes (bagging). Averaging cancels out the errors of individual trees. *Strengths:* strong on tabular data with mixed feature types, robust to outliers (e.g. extreme `adr` and `lead_time` kept in Stage 3), little tuning needed. *Weakness:* larger and slower, and less transparent than a single tree.

**Gradient Boosting (HistGradientBoosting).** Builds trees one after another, with each new tree correcting the mistakes of the previous ones. The "histogram" version bins numeric features, which makes it fast on 90k+ rows. *Strengths:* usually the most accurate on structured data, captures complex interactions. *Weakness:* more sensitive to settings, which is why Stage 6 tunes it carefully.

**Why these fit the business problem.** The data is tabular with mixed numeric and categorical features, non-linear relationships (Stage 4 showed deposit type and lead time interact strongly with cancellation) and a moderately imbalanced target (about 37% cancellations). The set deliberately spans simple and interpretable (Logistic Regression, Decision Tree) to more powerful (Random Forest, Boosting), so we can see whether extra complexity pays off.

## 5.6 Evaluation Metrics

Cancellations are the minority class, so accuracy alone can mislead. We report:

- **Accuracy**: overall share of correct predictions.
- **Precision**: of the bookings flagged as "will cancel", how many really cancel. Low precision means the hotel acts (e.g. overbooks) on false alarms.
- **Recall**: of the bookings that really cancel, how many we catch. Low recall means missed cancellations and empty rooms.
- **F1**: harmonic mean of precision and recall, summarising the trade-off at the default 0.5 threshold.
- **ROC-AUC**: how well the model ranks cancelling bookings above kept ones, independent of threshold.
- **PR-AUC (average precision)**: like ROC-AUC but focused on the minority class, so it is more informative with imbalance.

**Primary metrics for model choice: F1 and ROC-AUC.** The threshold can be adjusted later to suit the hotel's cost of a false alarm versus a missed cancellation (Stage 6).

## 5.7 Cross-Validated Model Comparison

Each model is trained and validated 5 times on the training data. We also record training-fold scores to detect **overfitting** (a large gap between training and validation scores).

In [ ]:
scoring = {"accuracy": "accuracy", "precision": "precision", "recall": "recall",
           "f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision"}

cv_results, cv_raw = [], {}
for name, pipe in models.items():
    t0 = time.time()
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring,
                            return_train_score=True, n_jobs=1)
    cv_raw[name] = scores
    row = {"Model": name}
    for m in scoring:
        row[f"{m} (mean)"] = scores[f"test_{m}"].mean()
        row[f"{m} (std)"] = scores[f"test_{m}"].std()
    row["train f1"] = scores["train_f1"].mean()
    row["train roc_auc"] = scores["train_roc_auc"].mean()
    row["fit time (s)"] = scores["fit_time"].mean()
    cv_results.append(row)
    print(f"{name:30s} done in {time.time()-t0:5.1f}s")

cv_df = pd.DataFrame(cv_results).set_index("Model")
show_cols = ["accuracy (mean)", "precision (mean)", "recall (mean)", "f1 (mean)", "f1 (std)",
             "roc_auc (mean)", "pr_auc (mean)", "fit time (s)"]
display(cv_df[show_cols].round(4))

In [ ]:
# Overfitting check: training vs validation score
gap = pd.DataFrame({
    "Train F1": cv_df["train f1"], "Validation F1": cv_df["f1 (mean)"],
    "Train ROC-AUC": cv_df["train roc_auc"], "Validation ROC-AUC": cv_df["roc_auc (mean)"],
})
gap["F1 gap"] = gap["Train F1"] - gap["Validation F1"]
gap["AUC gap"] = gap["Train ROC-AUC"] - gap["Validation ROC-AUC"]
display(gap.round(4))

# Spread of F1 and ROC-AUC across folds
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in zip(axes, ["f1", "roc_auc"], ["F1 score", "ROC-AUC"]):
    plot_df = pd.DataFrame({n: s[f"test_{metric}"] for n, s in cv_raw.items()})
    sns.boxplot(data=plot_df, orient="h", ax=ax, color="#8ecae6")
    ax.set_title(f"5-fold CV {title}")
    ax.set_xlabel(title)
plt.tight_layout()
plt.show()

## 5.8 Held-Out Test Set Evaluation

Each model is now fitted on the full training set and evaluated **once** on the unseen test set.

In [ ]:
fitted, test_rows, test_proba = {}, [], {}
for name, pipe in models.items():
    t0 = time.time()
    pipe.fit(X_train, y_train)
    fit_t = time.time() - t0
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    fitted[name], test_proba[name] = pipe, proba
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "Fit time (s)": fit_t,
    })
test_df = pd.DataFrame(test_rows).set_index("Model")
display(test_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name in list(models)[1:]:
    RocCurveDisplay.from_predictions(y_test, test_proba[name], name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, test_proba[name], name=name, ax=axes[1])
axes[0].plot([0, 1], [0, 1], "k--", alpha=.4, label="Chance")
axes[0].set_title("ROC curves (test set)")
axes[1].set_title("Precision-Recall curves (test set)")
axes[0].legend(loc="lower right"); axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for the non-baseline models
names = list(models)[1:]
fig, axes = plt.subplots(1, len(names), figsize=(4.2 * len(names), 4))
for ax, name in zip(axes, names):
    ConfusionMatrixDisplay.from_predictions(
        y_test, fitted[name].predict(X_test), display_labels=["Kept", "Cancelled"],
        cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()

## 5.9 Business Reading of the Errors

In this problem the two error types have different costs:

- **False negative** (a cancellation we miss): the room sits empty, revenue is lost.
- **False positive** (a booking flagged as cancelling that stays): if the hotel overbooks on this basis, it may have to relocate a guest, which costs money and goodwill.

The confusion matrices above show how many of each error type each model makes at the default 0.5 threshold. Because both costs are real, Stage 6 will study how moving the decision threshold changes this balance.

## 5.10 What Drives the Predictions? (Permutation Importance)

We measure how much the ROC-AUC on the test set drops when each original feature is randomly shuffled. A large drop means the model relies on that feature. Because this is applied to the whole pipeline, one-hot columns are grouped back into their original features.

In [ ]:
best_name = test_df["ROC-AUC"].drop("Baseline (majority class)").idxmax()
print("Highest test ROC-AUC:", best_name)

rng_idx = X_test.sample(6000, random_state=RANDOM_STATE).index
pi = permutation_importance(fitted[best_name], X_test.loc[rng_idx], y_test.loc[rng_idx],
                            scoring="roc_auc", n_repeats=3, random_state=RANDOM_STATE, n_jobs=1)
imp = (pd.DataFrame({"feature": X_test.columns, "importance": pi.importances_mean, "std": pi.importances_std})
         .sort_values("importance", ascending=False))

top = imp.head(15).iloc[::-1]
plt.figure(figsize=(8, 5.5))
plt.barh(top["feature"], top["importance"], xerr=top["std"], color="#219ebc")
plt.xlabel("Drop in ROC-AUC when shuffled")
plt.title(f"Top 15 features: permutation importance ({best_name})")
plt.tight_layout()
plt.show()
display(imp.head(10).round(4))

## 5.11 Robustness Check: Does the Model Work on Future Bookings?

A random split mixes 2015, 2016 and 2017 bookings. In practice a hotel predicts *future* bookings from *past* ones. As a stricter test, we train on arrivals up to 2016 and test on 2017 arrivals for every model.

In [ ]:
train_mask = data["arrival_date_year"] <= 2016
Xt, yt = X[train_mask], y[train_mask]
Xf, yf = X[~train_mask], y[~train_mask]
print(f"Train (2015-2016): {len(Xt)} rows | Test (2017): {len(Xf)} rows")

from sklearn.base import clone
time_rows = []
for name, pipe in models.items():
    p = clone(pipe).fit(Xt, yt)
    pred, proba = p.predict(Xf), p.predict_proba(Xf)[:, 1]
    time_rows.append({"Model": name,
                      "F1 (time split)": f1_score(yf, pred),
                      "ROC-AUC (time split)": roc_auc_score(yf, proba),
                      "F1 (random split)": test_df.loc[name, "F1"],
                      "ROC-AUC (random split)": test_df.loc[name, "ROC-AUC"]})
time_df = pd.DataFrame(time_rows).set_index("Model")
display(time_df.round(4))

## 5.12 Model Selection: Comparison and Justification

### Cross-validation and test results (all consistent)

| Model | CV F1 | CV ROC-AUC | Test F1 | Test ROC-AUC | Test Recall | Time-split ROC-AUC |
|---|---|---|---|---|---|---|
| Baseline (majority) | 0.000 | 0.500 | 0.000 | 0.500 | 0.000 | 0.500 |
| Logistic Regression | 0.583 | 0.855 | 0.583 | 0.856 | 0.509 | 0.833 |
| Decision Tree | 0.652 | 0.882 | 0.650 | 0.891 | 0.583 | 0.799 |
| Random Forest | 0.694 | 0.906 | 0.702 | 0.909 | 0.645 | 0.836 |
| **Gradient Boosting (HistGB)** | **0.701** | **0.911** | **0.702** | **0.914** | **0.662** | **0.844** |

**Key observations**

1. **Every real model beats the baseline.** The baseline reaches 72.4% accuracy by never predicting a cancellation, yet it catches none (recall 0). This confirms that accuracy alone is misleading and that F1 and ROC-AUC are the right primary metrics.
2. **Model complexity pays off.** Moving from Logistic Regression to tree ensembles lifts F1 from about 0.58 to 0.70 and ROC-AUC from 0.86 to 0.91. Cancellation depends on non-linear effects and interactions (deposit type, market segment, lead time, country) that a linear model cannot capture.
3. **Random Forest and Gradient Boosting are close, but Boosting is slightly better and more reliable.** It has the higher ROC-AUC and PR-AUC and the best recall (it catches more real cancellations). The Random Forest memorises the training folds (training F1 0.996 vs validation 0.694), whereas Boosting shows a small gap (0.723 vs 0.701). The Random Forest is also slower to train and the saved model is larger.
4. **Results are stable.** Fold-to-fold standard deviation of F1 is below 0.007 and the CV scores match the test scores, so the test set was not a lucky split.
5. **The time-based split is a harder test.** Training on 2015-2016 and testing on 2017 lowers every model, because booking behaviour changes over time. Gradient Boosting still has the best ROC-AUC (0.844). Logistic Regression has the best F1 on this split (0.628) because its threshold-0.5 predictions are less affected by the shift in cancellation rate, but its ranking ability (AUC) is lower. The Decision Tree degrades most, which shows its instability.

### Final choice: Gradient Boosting (HistGradientBoosting)

*Technical reasons:* highest ROC-AUC and PR-AUC in cross-validation, on the test set and on the time-based split; best recall; limited overfitting; fast training (about 4 seconds); handles mixed features and outliers without scaling.

*Business reasons:* it identifies about two thirds of cancellations at roughly 75% precision. For hotel revenue management this means most bookings flagged as risky really are, so overbooking or reminder actions are mostly well targeted, and fewer empty rooms go unforeseen. It outputs a probability, so the hotel can choose its own risk threshold.

*Trade-offs we accept:* it is less transparent than Logistic Regression or a single tree. We reduce this with permutation importance (above) in the report and in the Streamlit app. Logistic Regression remains a useful, explainable comparison model.

### Limitations and cautions to carry into Stage 6

- **`arrival_date_year`** ranks among the top features, but a model that depends on the calendar year cannot extrapolate to 2018 and later. Stage 6 should test removing it.
- **`country` and `agent`** are the most important features. They likely act as proxies for hotel policies and booking channels in this particular dataset, so the model may transfer poorly to a different hotel.
- **`required_car_parking_spaces` and `booking_changes`** may be updated after booking. If they are not known at prediction time, they should be removed before deployment. This should be confirmed with the data owner.
- **Duplicates were removed for modelling.** The cancellation rate in the modelling data is 27.6% versus 37.1% in the full data, so absolute precision/recall values would differ on real traffic. The model ranking should not change.
- Models use default hyperparameters, so the gap between Random Forest and Boosting may change after tuning.

In [ ]:
# Summary table used in the report
summary = pd.DataFrame({
    "CV F1": cv_df["f1 (mean)"], "CV ROC-AUC": cv_df["roc_auc (mean)"],
    "Test F1": test_df["F1"], "Test ROC-AUC": test_df["ROC-AUC"],
    "Test Precision": test_df["Precision"], "Test Recall": test_df["Recall"],
    "Time-split ROC-AUC": time_df["ROC-AUC (time split)"],
    "Fit time (s)": test_df["Fit time (s)"],
}).round(4)
display(summary)
summary.to_csv("stage5_model_comparison.csv")

# Save the shortlisted models for Stage 6 (tuning) and Stage 7 (deployment)
import os
os.makedirs("models", exist_ok=True)
FINAL_CANDIDATE = best_name
joblib.dump(fitted[FINAL_CANDIDATE], "models/stage5_final_candidate.joblib", compress=3)
print("Saved:", FINAL_CANDIDATE, "-> models/stage5_final_candidate.joblib")

## 5.13 Summary and Next Steps

**What we did in Stage 5**
- Found and removed a leakage source (`assigned_room_type`) and quantified how duplicate rows inflate scores (ROC-AUC 0.958 with duplicates vs 0.906 without).
- Built leakage-safe pipelines and compared a baseline plus four models with stratified 5-fold cross-validation, a held-out test set and a time-based split.
- Selected **Gradient Boosting (HistGradientBoosting)** as the final candidate, with Random Forest as the runner-up.

**Outputs**
- `stage5_model_comparison.csv`: summary table for the report.
- `models/stage5_final_candidate.joblib`: fitted pipeline for the final candidate (the preprocessing is included, so it accepts raw booking columns).

